# Recommender comparison on validation

Run `01_data`, `02_popularity` and `03_matrix_factorization` in validation mode first.
The setup checks out `feat/evaluation` for review. After merging, set `BRANCH = "main"`.
Restart the kernel when switching branches.

Both models use exactly the saved user sample and the same `evaluate_validation()`
protocol: K=10, at least 10 train ratings per candidate, and all training ratings
excluded from recommendations. Recall and binary NDCG use users with at least one
validation rating >= 4; every validation positive counts, including out-of-catalog
films. Beyond-accuracy metrics use the full sample. Novelty and long-tail membership
are computed from train only. Diversity is Jaccard distance between genre sets.

In [ ]:
import importlib
import os
import subprocess
import sys
import types

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    from IPython import get_ipython

    shell = get_ipython()
    BRANCH = "feat/content-based"
    REPO_URL = "https://github.com/Cxsee/AI-in-Industry-project.git"
    EXTRAS = ""
    DRIVE_DIR = "/content/drive/MyDrive/movierec"
    REPO_DIR = "/content/AI-in-Industry-project-evaluation"

    drive.mount("/content/drive")
    os.environ["MOVIEREC_DATA_DIR"] = f"{DRIVE_DIR}/data"
    os.environ["MOVIEREC_RESULTS_DIR"] = f"{DRIVE_DIR}/results"
    os.makedirs(os.environ["MOVIEREC_DATA_DIR"], exist_ok=True)
    os.makedirs(os.environ["MOVIEREC_RESULTS_DIR"], exist_ok=True)

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)

    shell.run_line_magic("cd", REPO_DIR)
    subprocess.run(["git", "fetch", "-q", REPO_URL, BRANCH], check=True)
    subprocess.run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    if not os.path.isfile(f"{REPO_DIR}/src/movierec/evaluation.py"):
        raise RuntimeError(f"Branch {BRANCH!r} has no evaluation module. Check BRANCH above.")
    target = f".[{EXTRAS}]" if EXTRAS else "."
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", target], check=True)

    # The running kernel doesn't pick up an editable install made after it started.
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")

    # Colab's IPython 7.34 still imports `imp`, which Python 3.13 removed.
    sys.modules.setdefault("imp", types.SimpleNamespace(reload=importlib.reload))
    shell.run_line_magic("load_ext", "autoreload")
    shell.run_line_magic("autoreload", "2")

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl
from IPython.display import HTML, display

from movierec import paths
from movierec.evaluation import evaluate_validation

DATA_DIR = paths.data_dir()
RESULTS_DIR = paths.results_dir()
K = 10
MIN_RATINGS = 10
RELEVANT_RATING = 4.0
MODELS = {"popularity": "Popularity", "mf": "Matrix Factorization"}
if (RESULTS_DIR / "content_val.parquet").is_file():
    MODELS["content"] = "Content-Based"
else:
    print("Content-Based not available yet; run 04_content_based to add it.")

inputs = {name: DATA_DIR / f"{name}.parquet" for name in ("ratings", "users_sample", "movies")}
ranking_paths = {model: RESULTS_DIR / f"{model}_val.parquet" for model in MODELS}
missing = [str(path) for path in [*inputs.values(), *ranking_paths.values()] if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Run 01_data, 02_popularity (val) and 03_matrix_factorization (val). Missing: "
        + ", ".join(missing)
    )

ratings = pl.scan_parquet(inputs["ratings"])
users = pl.read_parquet(inputs["users_sample"]).sort("sample_rank")
movies = pl.read_parquet(inputs["movies"])
user_metrics, summaries = {}, []
for model in MODELS:
    per_user, summary = evaluate_validation(
        ratings,
        users,
        movies,
        pl.read_parquet(ranking_paths[model]),
        k=K,
        min_ratings=MIN_RATINGS,
        relevant_rating=RELEVANT_RATING,
    )
    user_metrics[model] = per_user
    summaries.append(
        summary.with_columns(pl.lit(model).alias("model"), pl.lit("val").alias("split"))
    )
summary = pl.concat(summaries)

# Missing rankings remain in the saved cohort and are reported, never silently dropped.
reference = user_metrics["popularity"].select("userId", "n_positives")
assert all(
    frame.select("userId", "n_positives").equals(reference) for frame in user_metrics.values()
)

## Popularity and Matrix Factorization

The table reports user means and global catalog coverage. Novelty is in bits;
the other metrics lie between 0 and 1. The cohort diagnostics make missing rankings
and users excluded from accuracy visible for each model.

In [ ]:
display(
    summary.select(
        "model",
        "n_users",
        "n_accuracy_users",
        "n_excluded_no_positives",
        "n_empty_rankings",
        "catalog_size",
        "k",
        "min_ratings",
        "relevant_rating",
    )
)
labels = {
    "recall": f"Recall@{K}",
    "ndcg": f"NDCG@{K}",
    "novelty": f"Novelty@{K} (bits)",
    "genre_diversity": f"Diversity@{K} (genres)",
    "long_tail_share": f"Long-tail share@{K}",
    "catalog_coverage": "Catalog coverage",
}
results = {row["model"]: row for row in summary.iter_rows(named=True)}
comparison = pl.DataFrame(
    {
        "Metric": list(labels.values()),
        **{label: [results[model][key] for key in labels] for model, label in MODELS.items()},
    },
    schema_overrides=dict.fromkeys(MODELS.values(), pl.Float64),
)
display(comparison)

bars = []
for key, label in labels.items():
    if key == "novelty":
        continue
    bars.append(f"<b>{label}</b>")
    for model, model_label in MODELS.items():
        value = results[model][key]
        if value is None:
            continue
        bars.append(
            f'<div style="display:flex;align-items:center;gap:8px;margin:6px 0">'
            f'<span style="width:170px">{model_label}</span>'
            f'<progress max="1" value="{value}" style="width:240px"></progress>'
            f"<span>{value:.4f}</span></div>"
        )
display(HTML("".join(bars)))

In [ ]:
outputs = {
    "comparison_val_summary.parquet": summary,
    "comparison_val.parquet": comparison,
}
for model, frame in user_metrics.items():
    outputs[f"{model}_val_user_metrics.parquet"] = frame
    outputs[f"{model}_val_summary.parquet"] = summary.filter(pl.col("model") == model)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name, frame in outputs.items():
    output = RESULTS_DIR / name
    with TemporaryDirectory(dir=RESULTS_DIR) as temp_dir:
        temporary_output = Path(temp_dir) / name
        frame.write_parquet(temporary_output)
        paths.replace_file(temporary_output, output)
    print(f"Saved {output}")
comparison.write_csv(RESULTS_DIR / "comparison_val.csv")